# Preprocesamiento para el modelo - House Pricing

Ingeniería de características sobre las features seleccionadas en el análisis exploratorio (`pipeline_config.yaml`). Las transformaciones se implementan con `feature-engine` y se integran en un `Pipeline` de sklearn que se ajusta solo con los datos de entrenamiento. El resultado son los datasets procesados de entrenamiento, validación y prueba, que sirven de input para el notebook de modelado.

## 1. Librerías

Importación de las librerías para manipulación de datos, partición del dataset, transformaciones de `feature-engine`, construcción del pipeline y serialización.

In [1]:
import joblib
import numpy as np
import pandas as pd
import yaml
from feature_engine.creation import RelativeFeatures
from feature_engine.encoding import OrdinalEncoder
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from house_pricing.config import MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT, RAW_DATA_DIR

2026-11-03 19:07:39.381 | INFO     | house_pricing.config:<module>:11 - PROJ_ROOT path is: C:\Users\MO\Desktop\Business Intelligence\8vo. trimestre\Product_Development\Clase\House_pricing


## 2. Carga de configuración y datos

Lectura de la variable objetivo y de las features desde `pipeline_config.yaml`, y de los datasets crudos. Se incorpora `YrSold` como variable auxiliar, necesaria para calcular las antigüedades en la sección 5. La ruta y los nombres de archivo se definen en variables para poder cambiar la fuente sin modificar el resto del código.

In [2]:
CONFIG_PATH = PROJ_ROOT / "pipeline_config.yaml"
DATA_PATH = RAW_DATA_DIR
TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"

with open(CONFIG_PATH, encoding="utf-8") as f:
    pipeline_config = yaml.safe_load(f)

TARGET = pipeline_config["TARGET"]
MODEL_FEATURES = pipeline_config["MODEL_FEATURES"]
REFERENCE_YEAR = "YrSold"
INPUT_FEATURES = MODEL_FEATURES + [REFERENCE_YEAR]

train_raw = pd.read_csv(DATA_PATH / TRAIN_FILE)
test_raw = pd.read_csv(DATA_PATH / TEST_FILE)

print(f"Train: {train_raw.shape}")
print(f"Test: {test_raw.shape}")
print(f"Features del modelo: {len(MODEL_FEATURES)}")

Train: (1460, 81)
Test: (1459, 80)
Features del modelo: 30


## 3. Clasificación de variables

Agrupación de las features según el tratamiento que reciben, con el mismo criterio del análisis exploratorio (`DISCRETE_THRESHOLD` valores únicos):

- **Categóricas**: variables de texto más `MSSubClass`, que es un código de tipo de vivienda y no una cantidad.
- **Años**: `YearBuilt`, `YearRemodAdd` y `GarageYrBlt`, que se convierten en antigüedades.
- **Continuas**: el resto de numéricas con más de `DISCRETE_THRESHOLD` valores únicos.
- **Discretas**: el resto de numéricas.

In [3]:
DISCRETE_THRESHOLD = 15
YEAR_FEATURES = ["YearBuilt", "YearRemodAdd", "GarageYrBlt"]
NUMERIC_AS_CATEGORICAL = ["MSSubClass"]

categoricas = train_raw[MODEL_FEATURES].select_dtypes(include=["object", "str"]).columns.tolist() + NUMERIC_AS_CATEGORICAL
numericas = [col for col in MODEL_FEATURES if col not in categoricas]

n_unique = train_raw[numericas].nunique()
continuas = [col for col in numericas if n_unique[col] > DISCRETE_THRESHOLD and col not in YEAR_FEATURES]
discretas = [col for col in numericas if n_unique[col] <= DISCRETE_THRESHOLD]

print(f"Categóricas ({len(categoricas)}): {categoricas}")
print(f"Años ({len(YEAR_FEATURES)}): {YEAR_FEATURES}")
print(f"Continuas ({len(continuas)}): {continuas}")
print(f"Discretas ({len(discretas)}): {discretas}")

Categóricas (11): ['Neighborhood', 'ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'FireplaceQu', 'Foundation', 'GarageType', 'BsmtFinType1', 'HeatingQC', 'MSSubClass']
Años (3): ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
Continuas (7): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']
Discretas (9): ['OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces', 'OverallCond', 'HalfBath', 'BedroomAbvGr', 'BsmtFullBath']


## 4. División en entrenamiento y validación

Separación de `train.csv` en entrenamiento (80%) y validación (20%) antes de ajustar cualquier transformación, para que el pipeline aprenda solo del conjunto de entrenamiento y la validación refleje datos no vistos. La variable objetivo se transforma con `log1p` para reducir su sesgo; las predicciones del modelo se revierten con `np.expm1`.

In [4]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

X = train_raw[INPUT_FEATURES]
y = np.log1p(train_raw[TARGET])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
X_test = test_raw[INPUT_FEATURES]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Sesgo de {TARGET}: original = {train_raw[TARGET].skew():.2f} | log1p = {y.skew():.2f}")

X_train: (1168, 31) | X_val: (292, 31) | X_test: (1459, 31)
Sesgo de SalePrice: original = 1.88 | log1p = 0.12


## 5. Imputación de variables categóricas y numéricas

Reemplazo de valores faltantes: las categóricas con la **moda** (`CategoricalImputer`) y las numéricas con la **mediana** (`MeanMedianImputer`), calculadas solo con el conjunto de entrenamiento. Se aplican a todas las variables del grupo para cubrir también los faltantes que solo aparecen en prueba.

Cada paso se aplica de forma encadenada sobre `X_train` para revisar su efecto; el pipeline final de la sección 11 reúne los mismos transformadores.

In [5]:
categorical_imputer = CategoricalImputer(imputation_method="frequent", variables=categoricas, ignore_format=True)
numerical_imputer = MeanMedianImputer(imputation_method="median", variables=numericas)

print(f"Faltantes antes: {X_train.isna().sum().sum()}")
X_step = categorical_imputer.fit_transform(X_train, y_train)
X_step = numerical_imputer.fit_transform(X_step, y_train)
print(f"Faltantes después: {X_step.isna().sum().sum()}")

pd.Series(categorical_imputer.imputer_dict_, name="moda").to_frame()

Faltantes antes: 1018
Faltantes después: 0


,moda
Neighborhood,NAmes
ExterQual,TA
BsmtQual,TA
KitchenQual,TA
GarageFinish,Unf
FireplaceQu,Gd
Foundation,PConc
GarageType,Attchd
BsmtFinType1,Unf
HeatingQC,Ex


## 6. Conversión de años a antigüedad

Reemplazo de cada año por los años transcurridos hasta la venta (`YrSold − año`), que expresa de forma directa la antigüedad de la casa, de su remodelación y del garaje. `RelativeFeatures` crea las nuevas variables (`YrSold_sub_<año>`) y, con `drop_original=True`, elimina los años originales y `YrSold`, que solo se usa como referencia.

In [6]:
age_creator = RelativeFeatures(variables=[REFERENCE_YEAR], reference=YEAR_FEATURES, func=["sub"], drop_original=True)

X_step = age_creator.fit_transform(X_step, y_train)

edades = [f"{REFERENCE_YEAR}_sub_{col}" for col in YEAR_FEATURES]
continuas_modelo = continuas + edades

X_step[edades].describe().round(1)

,YrSold_sub_YearBuilt,YrSold_sub_YearRemodAdd,YrSold_sub_GarageYrBlt
count,1168.0,1168.0,1168.0
mean,36.9,22.9,29.1
std,30.7,20.7,24.1
min,0.0,-1.0,0.0
25%,7.0,4.0,7.0
50%,35.0,14.0,28.0
75%,55.0,42.0,45.0
max,136.0,60.0,107.0


## 7. Codificación de variables categóricas

Codificación ordinal guiada por el target (`OrdinalEncoder` con `encoding_method="ordered"`): las categorías de cada variable se numeran de 0 en adelante según la media de `SalePrice` (en log) que tienen en entrenamiento, de modo que el número refleja el nivel de precio asociado. Las categorías que no existen en entrenamiento, como `MSSubClass = 150` en prueba, se codifican como `-1`.

In [7]:
encoder = OrdinalEncoder(encoding_method="ordered", variables=categoricas, ignore_format=True, unseen="encode")

X_step = encoder.fit_transform(X_step, y_train)

pd.Series(encoder.encoder_dict_["Neighborhood"], name="codigo").sort_values().to_frame().T

,MeadowV,BrDale,IDOTRR,BrkSide,Edwards,Blueste,OldTown,Sawyer,SWISU,NAmes,...,Gilbert,CollgCr,Crawfor,ClearCr,Somerst,Timber,Veenker,StoneBr,NridgHt,NoRidge
codigo,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24


## 8. Tratamiento de outliers

Recorte de valores extremos en las variables continuas con `Winsorizer`: los valores fuera de `Q1 − 1.5·IQR` y `Q3 + 1.5·IQR` se reemplazan por el límite correspondiente, sin eliminar filas. Los límites se calculan con el conjunto de entrenamiento.

In [8]:
winsorizer = Winsorizer(capping_method="iqr", tail="both", fold=1.5, variables=continuas_modelo)

X_capped = winsorizer.fit_transform(X_step, y_train)

limites = pd.DataFrame({
    "limite_inferior": winsorizer.left_tail_caps_,
    "limite_superior": winsorizer.right_tail_caps_,
    "valores_recortados": (X_capped[continuas_modelo] != X_step[continuas_modelo]).sum(),
}).round(1)
X_step = X_capped
limites

,limite_inferior,limite_superior,valores_recortados
GrLivArea,176.4,2761.4,23
GarageArea,-11.5,928.5,18
TotalBsmtSF,41.1,2054.1,47
1stFlrSF,124.9,2159.9,13
MasVnrArea,-249.0,415.0,77
BsmtFinSF1,-1081.5,1802.5,5
LotFrontage,31.5,107.5,85
YrSold_sub_YearBuilt,-65.0,127.0,4
YrSold_sub_YearRemodAdd,-53.0,99.0,0
YrSold_sub_GarageYrBlt,-50.0,102.0,1


## 9. Transformación de variables continuas

Aplicación de la transformación **Yeo-Johnson** (`YeoJohnsonTransformer`) a las variables continuas con sesgo alto (|skew| > `SKEW_THRESHOLD` en los datos crudos de entrenamiento). Yeo-Johnson estima la potencia óptima por variable y admite ceros, presentes en `MasVnrArea`, `BsmtFinSF1` y `TotalBsmtSF`.

In [9]:
SKEW_THRESHOLD = 0.75

skew_raw = X_train[continuas].skew()
continuas_sesgadas = skew_raw[skew_raw.abs() > SKEW_THRESHOLD].index.tolist()

yeo_johnson = YeoJohnsonTransformer(variables=continuas_sesgadas)

skew_antes = X_step[continuas_sesgadas].skew()
X_step = yeo_johnson.fit_transform(X_step, y_train)

pd.DataFrame({
    "sesgo_antes": skew_antes,
    "sesgo_despues": X_step[continuas_sesgadas].skew(),
    "lambda": pd.Series(yeo_johnson.lambda_dict_),
}).round(2)

,sesgo_antes,sesgo_despues,lambda
GrLivArea,0.59,-0.01,0.22
TotalBsmtSF,0.24,0.05,0.89
1stFlrSF,0.61,-0.00,0.14
MasVnrArea,1.27,0.37,-0.22
BsmtFinSF1,0.73,-0.42,0.23
LotFrontage,0.06,0.01,0.94


## 10. Escalado de variables

Estandarización de todas las features con `StandardScaler` (media 0 y desviación estándar 1), envuelto en `SklearnTransformerWrapper` para conservar el formato `DataFrame` y los nombres de columna.

In [10]:
scaler = SklearnTransformerWrapper(transformer=StandardScaler())

X_step = scaler.fit_transform(X_step, y_train)

X_step.describe().loc[["mean", "std"]].round(2)

,GrLivArea,GarageArea,TotalBsmtSF,1stFlrSF,MasVnrArea,BsmtFinSF1,LotFrontage,OverallQual,GarageCars,FullBath,...,KitchenQual,GarageFinish,FireplaceQu,Foundation,GarageType,BsmtFinType1,HeatingQC,YrSold_sub_YearBuilt,YrSold_sub_YearRemodAdd,YrSold_sub_GarageYrBlt
mean,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0,-0.0,...,-0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 11. Pipeline de preprocesamiento

Integración de todos los transformadores en un `Pipeline` de sklearn, en el orden de las secciones anteriores. El pipeline se ajusta únicamente con el conjunto de entrenamiento y se aplica a validación y prueba, lo que garantiza las mismas transformaciones en todos los conjuntos y evita fuga de información.

In [11]:
preprocessing_pipeline = Pipeline(steps=[
    ("categorical_imputer", categorical_imputer),
    ("numerical_imputer", numerical_imputer),
    ("age_creator", age_creator),
    ("encoder", encoder),
    ("winsorizer", winsorizer),
    ("yeo_johnson", yeo_johnson),
    ("scaler", scaler),
])

preprocessing_pipeline.fit(X_train, y_train)

X_train_processed = preprocessing_pipeline.transform(X_train)
X_val_processed = preprocessing_pipeline.transform(X_val)
X_test_processed = preprocessing_pipeline.transform(X_test)

print(f"Train: {X_train_processed.shape} | Val: {X_val_processed.shape} | Test: {X_test_processed.shape}")
print(f"Faltantes -> Train: {X_train_processed.isna().sum().sum()} | Val: {X_val_processed.isna().sum().sum()} | Test: {X_test_processed.isna().sum().sum()}")
preprocessing_pipeline

Train: (1168, 30) | Val: (292, 30) | Test: (1459, 30)
Faltantes -> Train: 0 | Val: 0 | Test: 0


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('categorical_imputer', ...), ('numerical_imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,31
,imputation_method,'frequent'
,variables,"['Neighborhood', 'ExterQual', ...]"
,ignore_format,True
,fill_value,'Missing'


## 12. Exportación de datasets y pipeline

Almacenamiento de los resultados para el notebook de modelado:

- `data/processed/train_processed.csv` y `val_processed.csv`: features procesadas más `SalePrice` en escala log.
- `data/processed/test_processed.csv`: features procesadas más `Id`, necesario para generar las predicciones finales.
- `models/preprocessing_pipeline.pkl`: pipeline ajustado, para transformar datos nuevos antes de predecir.

In [12]:
TRAIN_OUTPUT = PROCESSED_DATA_DIR / "train_processed.csv"
VAL_OUTPUT = PROCESSED_DATA_DIR / "val_processed.csv"
TEST_OUTPUT = PROCESSED_DATA_DIR / "test_processed.csv"
PIPELINE_OUTPUT = MODELS_DIR / "preprocessing_pipeline.pkl"

X_train_processed.assign(**{TARGET: y_train}).to_csv(TRAIN_OUTPUT, index=False)
X_val_processed.assign(**{TARGET: y_val}).to_csv(VAL_OUTPUT, index=False)
X_test_processed.assign(Id=test_raw["Id"]).to_csv(TEST_OUTPUT, index=False)
joblib.dump(preprocessing_pipeline, PIPELINE_OUTPUT)

for path in [TRAIN_OUTPUT, VAL_OUTPUT, TEST_OUTPUT, PIPELINE_OUTPUT]:
    print(f"Guardado: {path}")

Guardado: C:\Users\MO\Desktop\Business Intelligence\8vo. trimestre\Product_Development\Clase\House_pricing\data\processed\train_processed.csv
Guardado: C:\Users\MO\Desktop\Business Intelligence\8vo. trimestre\Product_Development\Clase\House_pricing\data\processed\val_processed.csv
Guardado: C:\Users\MO\Desktop\Business Intelligence\8vo. trimestre\Product_Development\Clase\House_pricing\data\processed\test_processed.csv
Guardado: C:\Users\MO\Desktop\Business Intelligence\8vo. trimestre\Product_Development\Clase\House_pricing\models\preprocessing_pipeline.pkl
